### Frozen Lake Tutorial

Source: https://www.youtube.com/watch?v=EUrWGTCGzlA&t=972s

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
# Define model
class DQN(nn.Module):
    def __init__(self, in_states, h1_nodes, out_actions):
        super().__init__()

        # define network layers.
        self.fc1 = nn.Linear(in_states, h1_nodes)
        self.out = nn.Linear(h1_nodes, out_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.out(x)
        return x

In [3]:
# Define memory for replay experience.
class ReplayMemory():
    def __init__(self, maxlen):
        self.memory = deque([], maxlen=maxlen)

    def append(self, transition):
        self.memory.append(transition)
    
    def sample(self, sample_size):
        return random.sample(self.memory, sample_size)

    def __len__(self):
        return len(self.memory)


In [10]:

class FrozenLakeDQL():

    learning_rate_a = 0.001
    discount_factor_g = 0.9
    network_sync_policy = 10  # Number of steps the agent takes before syncing the policy and the target network.
    memory_replay_size = 1000
    mini_batch_size = 32  # Size of the training dataset sampled from the replay memory.

    # Neural nets
    loss_fn = nn.MSELoss()
    optimizer = None  # Initialize later.
    ACTIONS = ["L", "D", "R", "U"]  # For printing 0,1,2,3 -> Left, Down, Right, Up

    # Train
    def train(self, episodes, render=False, is_slippery=False):

        # Create FrozenLake instance
        env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=is_slippery, render_mode="human" if render else None)
        num_states = env.observation_space.n
        num_actions = env.action_space.n

        epsilon = 1  # Start with full exploration.
        memory = ReplayMemory(self.memory_replay_size)

        # Create identical policy and target network to start.
        policy_dqn = DQN(in_states=num_states, h1_nodes=num_states, out_actions=num_actions)
        target_dqn = DQN(in_states=num_states, h1_nodes=num_states, out_actions=num_actions)

        # Copy weights from the policy network into the tarter network.
        target_dqn.load_state_dict(policy_dqn.state_dict())

        print("Policy (random, before training):")
        self.print_dqn(policy_dqn)

        self.optimizer = torch.optim.Adam(policy_dqn.parameters(), lr=self.learning_rate_a)

        # List to keep track of rewards collected per episode. Initialize to zeros.
        rewards_per_episode = np.zeros(episodes)

        # List to keep track of the epsilon decay.
        epsilon_history = []

        # Track the number of steps taken. Used for syncing policy -> target network.
        step_counter = 0

        for i in range(episodes):
            state = env.reset()[0]
            terminated = False  # True when the agent falls in a hole or reaches its goal.
            truncated = False  # True when the agent takes more than 200 actions.

            # Agent navigates the map until it falls into a hole or reaches its goal. Or has taken 200 steps.
            while(not terminated and not truncated):

                # Epsilon-greedy action selection.
                if random.random() < epsilon:
                    # Select a random action.
                    action = env.action_space.sample()  # Actions are: 0, 1, 2, 3 -> Left, Down, Right, Up
                else:
                    # Select the best action.
                    with torch.no_grad():
                        action = policy_dqn(self.state_to_dqn_input(state, num_states)).argmax().item()
                    
                # Execute action:
                new_state, reward, terminated, truncated, _ = env.step(action)

                # Save experience into memory.
                memory.append((state, action, new_state, reward, terminated))

                # Move to the next state:
                state = new_state

                step_counter += 1

            # Keep track of the rewards collected per episode.
            if reward == 1:
                rewards_per_episode[i] = 1
            
            # Check if enough experience has been collected and if at least 1 reward has been collected.
            if len(memory) > self.mini_batch_size and np.sum(rewards_per_episode) > 0:
                
                # Sample a mini-batch from the replay memory.
                mini_batch = memory.sample(self.mini_batch_size)
                self.optimize(mini_batch, policy_dqn, target_dqn)

                # Decay epsilon
                epsilon = max(epsilon - 1 / episodes, 0)
                epsilon_history.append(epsilon)

                # Copy the policy network to the target network after a certain number of steps.
                if step_counter > self.network_sync_policy:
                    target_dqn.load_state_dict(policy_dqn.state_dict())
                    step_counter = 0

        # Close the environment
        env.close()

        # Save the policy
        torch.save(policy_dqn.state_dict(), "frozen_lake_dqn.pt")

        # Create new graph
        plt.figure(1)

        # Plot the average rewards (y-axis) vs episodes (x-axis)
        sum_rewards = np.zeros(episodes)
        for x in range(episodes):
            sum_rewards[x] = np.sum(rewards_per_episode[max(0, x - 100):(x+1)])
        plt.subplot(121)
        plt.plot(sum_rewards)

        # PLot the epsilon decay
        plt.subplot(122)
        plt.plot(epsilon_history)

        plt.savefig("frozen_lake_dqn.png")



    def optimize(self, mini_batch, policy_dqn, target_dqn):

        # Get the number of nodes.
        num_states = policy_dqn.fc1.in_features

        current_q_list = []
        target_q_list = []

        for state, action, new_state, reward, terminated in mini_batch:
            #Apply the DQL value formula.
            if terminated:
                # When in a terminated state, target q value should be set to the reward.
                target = torch.FloatTensor([reward])
            else:
                # Calculate the target q value.
                with torch.no_grad():
                    target = torch.FloatTensor(
                        reward + self.discount_factor_g * target_dqn(self.state_to_dqn_input(new_state, num_states)).max()
                    )
            
            # Get the current set of q values.
            current_q = policy_dqn(self.state_to_dqn_input(state, num_states))
            current_q_list.append(current_q)

            # Get the target set of Q values
            target_q = target_dqn(self.state_to_dqn_input(state, num_states))

            # Adjust the specific action to the target that was just calculated
            target_q[action] = target
            target_q_list.append(target_q)

        # Compute the loss for the whole mini batch.
        loss = self.loss_fn(torch.stack(current_q_list), torch.stack(target_q_list))

        # Optimize the model.
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
    
    def state_to_dqn_input(self, state:int, num_states:int) -> torch.Tensor:
        input_tensor = torch.zeros(num_states)
        input_tensor[state] = 1
        return input_tensor

    # Run the environment with the learned policy.
    def test(self, episodes, is_slippery=False):

        env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=is_slippery, render_mode="human")
        num_states = env.observation_space.n
        num_actions = env.action_space.n

        # Load the learned policy.
        policy_dqn = DQN(in_states=num_states, h1_nodes=num_states, out_actions=num_actions)
        policy_dqn.load_state_dict(torch.load("frozen_lake_dqn.pt"))
        policy_dqn.eval()  # Switch model to evaluation mode.

        print("Trained policy:")
        self.print_dqn(policy_dqn)

        for i in range(episodes):
            state = env.reset()[0]
            terminated = False
            truncated = False
        
        while(not terminated and not truncated):
            with torch.no_grad():
                action = policy_dqn(self.state_to_dqn_input(state, num_states)).argmax().item()
            
            # Execute action.
            state, reward, terminated, truncated, _ = env.step(action)
        
        env.close()

    def print_dqn(self, dqn):
        # Get number of input nodes
        num_states = dqn.fc1.in_features

        # Loop each state and print policy to console
        for s in range(num_states):
            #  Format q values for printing
            q_values = ''
            for q in dqn(self.state_to_dqn_input(s, num_states)).tolist():
                q_values += "{:+.2f}".format(q)+' '  # Concatenate q values, format to 2 decimals
            q_values=q_values.rstrip()              # Remove space at the end

            # Map the best action to L D R U
            best_action = self.ACTIONS[dqn(self.state_to_dqn_input(s, num_states)).argmax()]

            # Print policy in the format of: state, action, q values
            # The printed layout matches the FrozenLake map.
            print(f'{s:02},{best_action},[{q_values}]', end=' ')         
            if (s+1)%4==0:
                print("\n") # Print a newline every 4 states





In [12]:
%%time
frozen_lake = FrozenLakeDQL()
is_slippery = False
#frozen_lake.train(2000, is_slippery=is_slippery)
frozen_lake.test(4, is_slippery=is_slippery)

Trained policy:
00,D,[+0.53 +0.59 +0.59 +0.53] 01,R,[+0.54 +0.00 +0.66 +0.60] 02,D,[+0.59 +0.73 +0.59 +0.66] 03,L,[+0.66 -0.00 +0.59 +0.42] 

04,D,[+0.58 +0.66 +0.05 +0.51] 05,U,[+0.39 +0.44 +0.35 +0.54] 06,D,[+0.00 +0.81 -0.01 +0.65] 07,D,[+0.40 +0.45 +0.42 +0.36] 

08,R,[+0.64 +0.01 +0.73 +0.57] 09,D,[+0.66 +0.81 +0.79 +0.03] 10,D,[+0.73 +0.90 -0.01 +0.73] 11,D,[+0.45 +0.61 +0.32 +0.57] 

12,U,[+0.54 +0.35 +0.56 +0.65] 13,R,[+0.00 +0.81 +0.90 +0.73] 14,R,[+0.81 +0.90 +1.00 +0.82] 15,U,[+0.56 +0.20 +0.43 +0.64] 

CPU times: total: 125 ms
Wall time: 2.64 s
